# Smart Nyuki — Queen Status from Hive Audio (TinyML)
Train a small CNN that listens to hive sound and predicts **queen present vs queen absent**, then shrink it to an int8 TensorFlow Lite model for the ESP32.

**Pipeline:** download audio → explore → cut 2 s windows → log-Mel features → split by recording → baseline (MFCC + Random Forest) → CNN → int8 quantization → C header for TFLite Micro.

> Runtime: **Runtime → Change runtime type → T4 GPU** (CPU also works, just slower). Feature extraction for ~4,000 clips takes a few minutes.

In [ ]:
!pip -q install librosa kagglehub soundfile joblib

In [ ]:
import os, glob, json, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import tensorflow as tf
from joblib import Parallel, delayed
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Feature settings — the ESP32 must reproduce these EXACTLY
SR = 8000            # bee sound energy sits well below 4 kHz; 8 kHz keeps the device cheap
SEG_SECONDS = 2.0    # literature reports 1–3 s segments are enough
SEG_LEN = int(SR * SEG_SECONDS)
N_FFT, HOP = 512, 256
N_MELS, FMIN, FMAX = 40, 50, 4000
MAX_FILES = None     # set e.g. 600 for a quick test run
print(tf.__version__, librosa.__version__)

## 1. Download the dataset
Kaggle **BeeHive Audio Dataset (with / without queen)**. If `kagglehub` asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab **Secrets** (key icon on the left) or upload `kaggle.json`.

In [ ]:
import kagglehub
DATA_DIR = kagglehub.dataset_download("harshkumar1711/beehive-audio-dataset-with-queen-and-without-queen")
print("Data in:", DATA_DIR)

AUDIO_EXT = ('.wav', '.mp3', '.flac', '.ogg')
for root, dirs, files in os.walk(DATA_DIR):
    depth = os.path.relpath(root, DATA_DIR).count(os.sep)
    if depth <= 3:
        n = sum(f.lower().endswith(AUDIO_EXT) for f in files)
        print('  ' * depth + os.path.basename(root) + f'/  ({n} audio files)')

## 2. Build the file list and labels
Labels come from folder / file names. Check the tree printed above — if the folder names differ, edit `NEG_KEYS` / `POS_KEYS`.
We use **`queen_absent = 1`** because that is the alarm we want to raise.

Note: we match on the path *relative to* `DATA_DIR`, because the dataset's own folder name contains both "with" and "without".

In [ ]:
NEG_KEYS = ('without', 'absent', 'noqueen', 'no_queen', 'queenless')   # -> queen absent
POS_KEYS = ('with', 'present', 'queen')                               # -> queen present (checked after NEG)

def label_from_rel(rel):
    s = rel.lower().replace('-', '_').replace(' ', '_')
    if any(k in s for k in NEG_KEYS): return 1
    if any(k in s for k in POS_KEYS): return 0
    return None

rows = []
for p in glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True):
    if p.lower().endswith(AUDIO_EXT):
        rel = os.path.relpath(p, DATA_DIR)
        y = label_from_rel(rel)
        if y is not None:
            rows.append((p, rel, y, os.path.basename(p), os.path.getsize(p)))

df = pd.DataFrame(rows, columns=['path', 'rel', 'queen_absent', 'fname', 'bytes'])
before = len(df)
df = df.drop_duplicates(subset=['fname', 'bytes']).reset_index(drop=True)   # Kaggle uploads sometimes duplicate folders
print(f'{before} files found, {len(df)} after removing duplicates')
if MAX_FILES:
    df = df.groupby('queen_absent', group_keys=False).apply(lambda g: g.sample(min(len(g), MAX_FILES // 2), random_state=SEED)).reset_index(drop=True)
print(df.queen_absent.value_counts().rename({0: 'queen present', 1: 'queen absent'}))
assert df.queen_absent.nunique() == 2, "Only one class found — edit NEG_KEYS / POS_KEYS to match the folder names"
df.sample(5, random_state=SEED)[['rel', 'queen_absent']]

## 3. Explore the sound
`center=False` in the STFT is deliberate: the ESP32 will slide a window over raw samples with no padding, so training must do the same.

In [ ]:
def load(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    return y

def logmel(y):
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
                                       fmin=FMIN, fmax=FMAX, power=2.0, center=False)
    return librosa.power_to_db(m, ref=1.0, amin=1e-10, top_db=None).astype(np.float32)

fig, axes = plt.subplots(2, 2, figsize=(15, 7))
for i, (lab, name) in enumerate([(0, 'Queen present'), (1, 'Queen absent')]):
    p = df[df.queen_absent == lab].sample(1, random_state=SEED).path.iloc[0]
    y = load(p)
    librosa.display.waveshow(y, sr=SR, ax=axes[0, i]); axes[0, i].set_title(name)
    img = librosa.display.specshow(logmel(y), sr=SR, hop_length=HOP, x_axis='time', y_axis='mel',
                                   fmin=FMIN, fmax=FMAX, ax=axes[1, i])
fig.colorbar(img, ax=axes[1, :], format='%+2.0f dB')
plt.show()

In [ ]:
# Average power spectrum per class — where do the two colonies differ?
def mean_spectrum(paths):
    acc = []
    for p in paths:
        y = load(p)
        acc.append((np.abs(librosa.stft(y, n_fft=2048, hop_length=1024)) ** 2).mean(axis=1))
    return 10 * np.log10(np.mean(acc, axis=0) + 1e-12)

freqs = librosa.fft_frequencies(sr=SR, n_fft=2048)
plt.figure(figsize=(12, 4))
for lab, name in [(0, 'Queen present'), (1, 'Queen absent')]:
    paths = df[df.queen_absent == lab].sample(min(100, (df.queen_absent == lab).sum()), random_state=SEED).path
    plt.plot(freqs, mean_spectrum(paths), label=name)
plt.xlim(0, 2000); plt.xlabel('Hz'); plt.ylabel('dB'); plt.legend(); plt.title('Mean spectrum (100 clips per class)')
plt.show()

## 4. Cut 2 s windows and extract features
Each 10 s clip becomes 5 windows. We keep the **source-file id** (`G`) for every window so we can split by recording later.

In [ ]:
def file_features(path):
    try:
        y = load(path)
    except Exception as e:
        return []
    feats = []
    for i in range(len(y) // SEG_LEN):
        seg = y[i * SEG_LEN:(i + 1) * SEG_LEN]
        if np.sqrt(np.mean(seg ** 2)) < 1e-4:      # skip silent / dead-mic windows
            continue
        feats.append(logmel(seg))
    return feats

results = Parallel(n_jobs=-1, verbose=5)(delayed(file_features)(p) for p in df.path)

X, Y, G = [], [], []
for gid, (feats, lab) in enumerate(zip(results, df.queen_absent)):
    for f in feats:
        X.append(f); Y.append(lab); G.append(gid)
X = np.stack(X)[..., None]          # (windows, 40 mel bands, frames, 1)
Y = np.array(Y, dtype=np.float32); G = np.array(G)
print('X', X.shape, '| windows per class:', np.bincount(Y.astype(int)))
np.savez_compressed('features.npz', X=X, Y=Y, G=G)   # cache so you can skip this step next time

## 5. Split by recording — not by window
If windows from the same 10 s clip land in both train and test, the model memorises the clip and the score is inflated.
`GroupShuffleSplit` keeps every clip entirely in one split.

**Bigger caveat:** if each class was recorded in different hives, the model may learn *which hive / microphone* it is hearing rather than *whether there is a queen*. If the dataset has hive or date info, group by that instead. The real test is recordings from your own Smart Nyuki hives.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr, tmp = next(gss.split(X, Y, G))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_i, te_i = next(gss2.split(X[tmp], Y[tmp], G[tmp]))
va, te = tmp[va_i], tmp[te_i]
assert not (set(G[tr]) & set(G[te])) and not (set(G[tr]) & set(G[va]))
print(f'train {len(tr)}  val {len(va)}  test {len(te)} windows')

# Per-mel-band normalisation from TRAIN only (save these for the ESP32)
mu = X[tr].mean(axis=(0, 2, 3), keepdims=True)
sd = X[tr].std(axis=(0, 2, 3), keepdims=True) + 1e-6
Xn = ((X - mu) / sd).astype(np.float32)

## 6. Baseline: MFCC statistics + Random Forest (Week 6 — supervised learning)
A simple model first. If the CNN cannot beat this, something is wrong.

In [ ]:
def mfcc_stats(Xdb):
    out = []
    for s in Xdb[..., 0]:
        m = librosa.feature.mfcc(S=s, n_mfcc=13)
        out.append(np.concatenate([m.mean(axis=1), m.std(axis=1)]))
    return np.array(out)

F = mfcc_stats(X)
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED).fit(F[tr], Y[tr])
p_rf = rf.predict_proba(F[te])[:, 1]
print(classification_report(Y[te], p_rf > 0.5, target_names=['queen present', 'queen absent']))
print('ROC-AUC:', round(roc_auc_score(Y[te], p_rf), 3))

## 7. Small CNN (sized for the ESP32)
Three conv blocks and global pooling — only a few thousand parameters. Light SpecAugment (frequency and time masks) helps it generalise to new hives.

In [ ]:
T = Xn.shape[2]

def spec_augment(x, y):
    f = tf.random.uniform([], 0, 6, dtype=tf.int32)
    f0 = tf.random.uniform([], 0, N_MELS - f, dtype=tf.int32)
    fmask = tf.concat([tf.ones([f0, 1, 1]), tf.zeros([f, 1, 1]), tf.ones([N_MELS - f0 - f, 1, 1])], axis=0)
    t = tf.random.uniform([], 0, 10, dtype=tf.int32)
    t0 = tf.random.uniform([], 0, T - t, dtype=tf.int32)
    tmask = tf.concat([tf.ones([1, t0, 1]), tf.zeros([1, t, 1]), tf.ones([1, T - t0 - t, 1])], axis=1)
    return x * fmask * tmask, y

BATCH = 64
train_ds = (tf.data.Dataset.from_tensor_slices((Xn[tr], Y[tr])).shuffle(len(tr), seed=SEED)
            .map(spec_augment, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((Xn[va], Y[va])).batch(BATCH)

def conv_block(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    return tf.keras.layers.MaxPool2D(2)(x)

inp = tf.keras.Input(shape=Xn.shape[1:])
x = conv_block(inp, 8)
x = conv_block(x, 16)
x = conv_block(x, 32)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()

hist = model.fit(train_ds, validation_data=val_ds, epochs=60, verbose=2, callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)])

pd.DataFrame(hist.history)[['loss', 'val_loss', 'accuracy', 'val_accuracy']].plot(figsize=(10, 4), grid=True); plt.show()

## 8. Evaluate — per window and per recording
On the hive we will average several windows before deciding, so the **per-recording** score is the one that matters.

In [ ]:
p_win = model.predict(Xn[te], verbose=0).ravel()
print('--- per 2 s window ---')
print(classification_report(Y[te], p_win > 0.5, target_names=['queen present', 'queen absent']))
print('ROC-AUC:', round(roc_auc_score(Y[te], p_win), 3))

rec = pd.DataFrame({'g': G[te], 'y': Y[te], 'p': p_win}).groupby('g').agg(y=('y', 'first'), p=('p', 'mean'))
print('--- per 10 s recording (mean of windows) ---')
print(classification_report(rec.y, rec.p > 0.5, target_names=['queen present', 'queen absent']))
ConfusionMatrixDisplay(confusion_matrix(rec.y, rec.p > 0.5), display_labels=['present', 'absent']).plot(cmap='YlOrBr')
plt.title('Per-recording confusion matrix'); plt.show()

## 9. Quantize to int8 TensorFlow Lite
Full-integer quantization so it runs on the ESP32 with TensorFlow Lite for Microcontrollers. The representative set calibrates the int8 ranges.

In [ ]:
def rep_data():
    for i in np.random.choice(tr, size=min(300, len(tr)), replace=False):
        yield [Xn[i:i + 1]]

def make_converter():
    try:
        return tf.lite.TFLiteConverter.from_keras_model(model)
    except Exception:
        model.export('queen_savedmodel')            # Keras 3 fallback
        return tf.lite.TFLiteConverter.from_saved_model('queen_savedmodel')

conv = make_converter()
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tflite_model = conv.convert()
open('queen_cnn_int8.tflite', 'wb').write(tflite_model)
print(f'int8 model: {len(tflite_model) / 1024:.1f} KB')

# Check the quantized model still agrees with the float one
interp = tf.lite.Interpreter(model_content=tflite_model); interp.allocate_tensors()
inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]
s_in, z_in = inp_d['quantization']; s_out, z_out = out_d['quantization']

def tflite_prob(x):
    q = np.clip(np.round(x / s_in + z_in), -128, 127).astype(np.int8)
    interp.set_tensor(inp_d['index'], q[None]); interp.invoke()
    return (interp.get_tensor(out_d['index'])[0, 0].astype(np.float32) - z_out) * s_out

p_q = np.array([tflite_prob(Xn[i]) for i in te])
print('float acc:', round(((p_win > 0.5) == Y[te]).mean(), 3), '| int8 acc:', round(((p_q > 0.5) == Y[te]).mean(), 3))
print('input quant  scale/zero:', s_in, z_in)

## 10. Export for the ESP32
- `queen_model.h` — the model as a C array for TFLite Micro
- `feature_config.json` — every feature setting, the Mel filterbank and the normalisation values. The firmware must apply them **in this order:** Hann-window 512 samples → FFT power → Mel filterbank → `10*log10(max(x, 1e-10))` → `(x - mu) / sd` → int8 quantize with the input scale/zero point.

In [ ]:
!xxd -i queen_cnn_int8.tflite > queen_model.h

mel_fb = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX)   # (40, 257)
cfg = dict(sr=SR, seg_seconds=SEG_SECONDS, n_fft=N_FFT, hop=HOP, n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
           window='hann', center=False, db_amin=1e-10, frames=int(T),
           mu=mu.ravel().tolist(), sd=sd.ravel().tolist(),
           input_scale=float(s_in), input_zero_point=int(z_in),
           output_scale=float(s_out), output_zero_point=int(z_out),
           mel_filterbank=mel_fb.tolist(), label_1='queen_absent')
json.dump(cfg, open('feature_config.json', 'w'))
model.save('queen_cnn.keras')

from google.colab import files
for f in ['queen_cnn_int8.tflite', 'queen_model.h', 'feature_config.json', 'queen_cnn.keras']:
    files.download(f)

## 11. Try it on a new recording + alarm logic
A single bad reading should not wake a beekeeper. Raise **"queen absent"** only after several recordings in a row score high (e.g. 3 consecutive recordings above 0.7). Tune on your own hives.

In [ ]:
def predict_recording(path):
    y = load(path)
    segs = [logmel(y[i * SEG_LEN:(i + 1) * SEG_LEN]) for i in range(len(y) // SEG_LEN)]
    x = ((np.stack(segs)[..., None] - mu) / sd).astype(np.float32)
    p = model.predict(x, verbose=0).ravel()
    return float(p.mean()), p

def queen_alarm(history, threshold=0.7, consecutive=3):
    """history: recent per-recording probabilities, newest last."""
    return len(history) >= consecutive and all(p > threshold for p in history[-consecutive:])

# Example — upload your own hive clip with the folder icon and set the path:
# prob, per_window = predict_recording('/content/my_hive.wav')
# print(f'P(queen absent) = {prob:.2f}', per_window.round(2))
sample = df.sample(1, random_state=7).iloc[0]
prob, per_window = predict_recording(sample.path)
print(sample.rel, '| true label:', 'absent' if sample.queen_absent else 'present', f'| P(absent) = {prob:.2f}')

## 12. From "queen present?" to "queen health"
This dataset only labels **present / absent**. It cannot tell a healthy queen from a failing one, detect a virgin queen, or warn of swarming. To get there:

1. **Collect your own labelled audio.** Record every Smart Nyuki hive on a schedule, and log what each inspection finds (queen seen, eggs/young brood present, queen cells, swarm). Inspection notes become labels.
2. **Add classes** — e.g. `healthy`, `queenless`, `queen cells / pre-swarm`, `newly requeened` — and change the last layer to `Dense(n_classes, activation='softmax')` with `sparse_categorical_crossentropy`.
3. **Fuse sensors.** Combine the audio probability with load-cell weight trends and temperature/humidity from the DHT11 in a small tabular model (the Week 6 regression/classification skills apply directly).
4. **Fine-tune** this model on your hives — transfer from the Kaggle data, then retrain the last layers on local recordings so it hears *your* microphones and *your* bees.